In [0]:
import requests
import json
import time
from datetime import datetime

# Get workspace URL and token
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
HOST = ctx.apiUrl().getOrElse(None)
TOKEN = ctx.apiToken().getOrElse(None)

HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

RATE_LIMIT_SLEEP = 0.3  # seconds between API calls

def api_get(path, params=None):
    """Make a GET request to the Databricks REST API."""
    url = f"{HOST}{path}"
    resp = requests.get(url, headers=HEADERS, params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()

def api_get_paginated(path, results_key, params=None):
    """Make paginated GET requests, collecting all items under results_key."""
    all_items = []
    page_token = None
    while True:
        p = dict(params or {})
        if page_token:
            p["page_token"] = page_token
        data = api_get(path, params=p)
        items = data.get(results_key, [])
        all_items.extend(items)
        page_token = data.get("next_page_token")
        if not page_token:
            break
        time.sleep(RATE_LIMIT_SLEEP)
    return all_items

print(f"Connected to: {HOST}")

In [0]:
# List all Genie spaces
spaces = api_get_paginated("/api/2.0/genie/spaces", "spaces")

print(f"Found {len(spaces)} Genie space(s):")
for s in spaces:
    print(f"  - {s.get('title', 'Untitled')} ({s.get('space_id')})")

# Build a lookup dict for space titles
space_lookup = {s["space_id"]: s.get("title", "Untitled") for s in spaces}

In [0]:
def safe_json(obj):
    """Serialize an object to JSON string, or return None."""
    if obj is None:
        return None
    return json.dumps(obj)

def ms_to_ts(epoch_ms):
    """Convert epoch milliseconds (int) to ISO-formatted string, or None."""
    if epoch_ms is None:
        return None
    try:
        return datetime.utcfromtimestamp(int(epoch_ms) / 1000).isoformat()
    except (ValueError, TypeError, OSError):
        return None

def extract_query_sql(attachments):
    """Extract the generated SQL from the attachments list."""
    if not attachments:
        return None
    for att in attachments:
        q = att.get("query")
        if q and q.get("query"):
            return q["query"]
    return None

def extract_text_response(attachments):
    """Extract the text response from the attachments list."""
    if not attachments:
        return None
    for att in attachments:
        t = att.get("text")
        if t and t.get("content"):
            return t["content"]
    return None

records = []
total_conversations = 0
total_messages = 0

for idx, space in enumerate(spaces):
    space_id = space["space_id"]
    space_title = space.get("title", "Untitled")
    print(f"\n[{idx+1}/{len(spaces)}] Processing space: {space_title} ({space_id})")

    # Fetch conversations for this space
    try:
        conversations = api_get_paginated(
            f"/api/2.0/genie/spaces/{space_id}/conversations",
            "conversations"
        )
        time.sleep(RATE_LIMIT_SLEEP)
    except requests.exceptions.HTTPError as e:
        print(f"  WARNING: Could not fetch conversations for space {space_id}: {e}")
        continue
    except Exception as e:
        print(f"  WARNING: Unexpected error for space {space_id}: {e}")
        continue

    print(f"  Found {len(conversations)} conversation(s)")
    total_conversations += len(conversations)

    for conv in conversations:
        conv_id = conv.get("conversation_id")
        conv_title = conv.get("title", "")
        conv_created = conv.get("created_timestamp")

        if not conv_id:
            print(f"  WARNING: Skipping conversation with no ID")
            continue

        # Fetch messages for this conversation
        try:
            messages = api_get_paginated(
                f"/api/2.0/genie/spaces/{space_id}/conversations/{conv_id}/messages",
                "messages"
            )
            time.sleep(RATE_LIMIT_SLEEP)
        except requests.exceptions.HTTPError as e:
            print(f"  WARNING: Could not fetch messages for conversation {conv_id}: {e}")
            continue
        except Exception as e:
            print(f"  WARNING: Unexpected error for conversation {conv_id}: {e}")
            continue

        total_messages += len(messages)

        for msg in messages:
            attachments = msg.get("attachments")
            records.append({
                "space_id": space_id,
                "space_title": space_title,
                "conversation_id": conv_id,
                "conversation_title": conv_title,
                "conversation_created_timestamp": ms_to_ts(conv_created),
                "message_id": msg.get("message_id"),
                "message_content": msg.get("content"),
                "message_status": msg.get("status"),
                "message_created_timestamp": ms_to_ts(msg.get("created_timestamp")),
                "message_last_updated_timestamp": ms_to_ts(msg.get("last_updated_timestamp")),
                "message_user_id": str(msg.get("user_id")) if msg.get("user_id") else None,
                "generated_sql": extract_query_sql(attachments),
                "text_response": extract_text_response(attachments),
                "attachments": safe_json(attachments),
            })

print(f"\n--- Summary ---")
print(f"Spaces processed: {len(spaces)}")
print(f"Conversations:    {total_conversations}")
print(f"Messages:         {total_messages}")
print(f"Records built:    {len(records)}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import col, to_timestamp

# Create schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS guido.genie")

if not records:
    print("No records to write. Table will not be created.")
else:
    # Create DataFrame from records
    df = spark.createDataFrame(records)

    # Convert ISO timestamp strings to proper TimestampType
    ts_cols = [
        "conversation_created_timestamp",
        "message_created_timestamp",
        "message_last_updated_timestamp",
    ]
    for c in ts_cols:
        df = df.withColumn(c, to_timestamp(col(c)))

    # Reorder columns for readability
    df = df.select(
        "space_id", "space_title",
        "conversation_id", "conversation_title", "conversation_created_timestamp",
        "message_id", "message_user_id", "message_content", "message_status",
        "message_created_timestamp", "message_last_updated_timestamp",
        "generated_sql", "text_response", "attachments"
    )

    # Write as Delta table (overwrite)
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("guido.genie.conversations")

    row_count = spark.table("guido.genie.conversations").count()
    print(f"Successfully wrote {row_count} rows to guido.genie.conversations")
    display(spark.table("guido.genie.conversations").limit(10))